In [1]:
from transformers import AutoModel, AutoTokenizer

In [2]:
all_text=list()
file=open("train_text.txt","r")

for line in file:
    all_text.append(line.split("\n")[0])
    
file.close()

all_label=list()
file=open("train_labels.txt","r")

for line in file:
    all_label.append(int(line.split("\n")[0]))

file.close()

train_set=all_text[0:10000]
train_label=all_label[0:10000]

dev_set=all_text[10000:12000]
dev_label=all_label[10000:12000]

test_set=all_text[12000:20000]
test_label=all_label[12000:20000]

In [3]:
import torch.nn as nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class BertClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.bert_model = AutoModel.from_pretrained("bert-base-cased")
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, token_ids):
        device = next(self.parameters()).device
        cls_rep = self.bert_model(token_ids['input_ids'].to(device)).pooler_output
        return self.classifier(cls_rep)



loss_function=nn.CrossEntropyLoss()      

In [4]:
model = BertClassifier(3).to(device)

from torch.optim import Adam

optimizer=Adam(model.parameters(),lr=0.000005)

bert_tokenizer=AutoTokenizer.from_pretrained("bert-base-cased")


In [5]:
n_epochs=5

import numpy as np
from sklearn.metrics import f1_score
from tqdm import tqdm


for epoch in range(n_epochs):
    all_loss=list()
    print("Epoch#",epoch+1," in progress")
    model.train()
    for i in tqdm(range(len(train_set))):
        model.zero_grad()
        sample=train_set[i]
        input_ids=bert_tokenizer(sample,add_special_tokens=True,return_tensors="pt")
        pred=model(input_ids)
        target=torch.tensor([train_label[i]],dtype=torch.long).to(device)
        loss=loss_function(pred,target)
        all_loss.append(loss.cpu().detach().item())
        loss.backward()
        optimizer.step()
    print("Average loss on training set=",np.mean(all_loss))

    with torch.no_grad():
        model.eval()
        all_pred=list()
        all_target=list()
        for i in range(len(dev_set)):
            sample=dev_set[i]
            input_ids=bert_tokenizer(sample,add_special_tokens=True,return_tensors="pt")
            pred=model(input_ids)
            pred=torch.argmax(pred)
            all_pred.append(pred.item())
            all_target.append(dev_label[i])
        print("F1 score on development set=",f1_score(all_target,all_pred,average="weighted"))



Epoch# 1  in progress


 68%|████████████████████████████████████████████████████▋                        | 6846/10000 [08:44<04:01, 13.06it/s]


KeyboardInterrupt: 

In [ ]:
with torch.no_grad():
    model.eval()
    all_pred=list()
    all_target=list()
    for i in range(len(test_set)):
        sample=test_set[i]
        input_ids=bert_tokenizer(sample,add_special_tokens=True,return_tensors="pt")
        pred=model(input_ids)
        pred=torch.argmax(pred)
        all_pred.append(pred.item())
        all_target.append(test_label[i])
        
                

from sklearn.metrics import classification_report

print("=====================")
print("Results for test set")
print("=====================")
print(classification_report(all_target,all_pred))
        
